In [1]:
%%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
except: get_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers>=4.55.3" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
!uv pip install transformers==4.55.4 pymongo vllm>=0.8.5
!uv pip install wandb -qU
!uv pip install weave -qU
!uv pip install titans-pytorch docling docling-core docling-ibm-models docling-parse pandas matplotlib genai tiktoken

In [2]:
import os
import pandas as pd

print("--- Accessing LongMemEval Datasets from Kaggle Input ---")

# In Kaggle, datasets are located in the /kaggle/input/ directory.
# The folder name matches the name you gave the dataset when you uploaded it.
dataset_path = "/kaggle/input/longmem/"

# Define the full paths to your CSV files
focused_csv_path = os.path.join(dataset_path, "cleaned_longmemeval_s_focused.csv")
full_csv_path = os.path.join(dataset_path, "cleaned_longmemeval_s_full.csv")

# --- Verification Step ---
# Let's quickly check if the files are where we expect them to be.
if os.path.exists(focused_csv_path) and os.path.exists(full_csv_path):
    print("✅ Datasets found successfully.")
    # We can do a quick test read to be sure
    try:
        df_test = pd.read_csv(focused_csv_path)
        print(f"Successfully read {len(df_test)} rows from the 'focused' dataset.")
    except Exception as e:
        print(f"❌ Error reading the CSV file: {e}")
else:
    print("❌ ERROR: Could not find the dataset files at the expected path.")
    print("Please check the 'Input' section on the right and make sure your dataset is named 'longmem'.")

--- Accessing LongMemEval Datasets from Kaggle Input ---
✅ Datasets found successfully.
Successfully read 306 rows from the 'focused' dataset.


In [3]:
import pandas as pd
from tqdm.notebook import tqdm
import os

# --- Create Results Directory ---
os.makedirs("results", exist_ok=True)

# --- Load Datasets from the correct Kaggle paths ---
# We use the 'focused_csv_path' and 'full_csv_path' variables we created in Cell 2.
print("--- Loading dataframes for inference ---")
focused_df = pd.read_csv(focused_csv_path)
full_df = pd.read_csv(full_csv_path)
print("✅ Dataframes loaded successfully.")



--- Loading dataframes for inference ---
✅ Dataframes loaded successfully.


In [4]:
from unsloth import FastLanguageModel
import torch
import torch.nn as nn
from titans_pytorch.neural_memory import NeuralMemory
from torch.amp import custom_fwd
import math
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()



# --- Titan-Reasoner Architecture (Must match the training script) ---

class ManualLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        gamma, beta = self.gamma, self.beta
        if gamma.ndim > 1:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return gamma * (x - mean) / (std + self.eps) + beta

class BatchedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        weight, bias = self.weight, self.bias
        if weight.ndim > 2:
            x = torch.einsum('...ni,...oi->...no', x, weight)
        else:
            x = torch.einsum('...i,oi->...o', x, weight)
        if bias is not None:
            if bias.ndim == x.ndim - 1:
                bias = bias.unsqueeze(-2)
            x = x + bias
        return x

class EagerMemoryMLP(nn.Module):
    def __init__(self, dim, mult=4, depth=1):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers.append(nn.Sequential(BatchedLinear(dim, dim * mult), nn.GELU(), BatchedLinear(dim * mult, dim)))
        self.model = nn.Sequential(*layers)
        self.norm = ManualLayerNorm(dim)

    def forward(self, x):
        return self.norm(self.model(x))

class PatchedNeuralMemory(NeuralMemory):
    def __init__(self, *args, **kwargs):
        dim = kwargs.get('dim')
        dim_head = kwargs.get('dim_head', dim)
        mlp_depth = kwargs.get('mem_mlp_depth', 1)
        eager_model = EagerMemoryMLP(dim=dim_head, depth=mlp_depth)
        kwargs['model'] = eager_model
        kwargs['mem_model_norm_add_residual'] = False
        kwargs['per_head_learned_parameters'] = False
        super().__init__(*args, **kwargs)
        self.store_norm = nn.LayerNorm(dim)
        self.retrieve_norm = nn.LayerNorm(dim)

class PatchedNeuralMemoryFP32(PatchedNeuralMemory):
    @custom_fwd(device_type='cuda', cast_inputs=torch.float32)
    def forward(self, *args, **kwargs):
        return super().forward(*args, **kwargs)

class TitanReasoner(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        model_dim = self.base_model.config.hidden_size
        print(f"--- Initializing Titans Neural Memory (dim={model_dim}) ---")
        self.memory = PatchedNeuralMemoryFP32(dim=model_dim, chunk_size=128, heads=4, dim_head=model_dim // 8)
        print("✅ Patched Neural Memory is online.")

    def forward(self, input_ids, attention_mask, labels=None, memory_state=None):
        input_embeds = self.base_model.get_input_embeddings()(input_ids)
        retrieved_memory, next_memory_state = self.memory(input_embeds, state=memory_state)
        augmented_embeds = input_embeds + retrieved_memory.to(input_embeds.dtype)
        outputs = self.base_model(inputs_embeds=augmented_embeds, attention_mask=attention_mask, labels=labels)
        return outputs, next_memory_state


# --- Configuration ---
TITAN_REASONER_REPO = "surfiniaburger/Purified-Reasoner-gpt-oss-20b-v1"
HF_TOKEN = user_secrets.get_secret("HUGGINGFACE_API_KEY") # Add your HF token, preferably from Kaggle Secrets

# --- Load the Model ---
print("--- Loading the Titan-Reasoner Model for Evaluation ---")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TITAN_REASONER_REPO,
    max_seq_length=2048, # Use a reasonable length for evaluation
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)
titan_reasoner_model = TitanReasoner(base_model).to("cuda")
memory_weights_path = hf_hub_download(repo_id=TITAN_REASONER_REPO, filename="titan_reasoner_memory.pt", token=HF_TOKEN)
titan_reasoner_model.memory.load_state_dict(torch.load(memory_weights_path, map_location="cuda"))
titan_reasoner_model.eval()

print("\n✅ Titan-Reasoner is loaded and ready for evaluation.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-09-26 13:42:57.087092: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758894177.308417     173 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758894177.367648     173 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 09-26 13:43:19 [__init__.py:216] Automatically detected platform cuda.
ERROR 09-26 13:43:20 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Unsloth: OpenAI failed to import - ignoring for now.
🦥 Unsloth Zoo will now patch everything to make training faster!
--- Loading the Titan-Reasoner Model for Evaluation ---
==((====))==  Unsloth 2025.9.8: Fast Gpt_Oss patching. Transformers: 4.55.4. vLLM: 0.10.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/31.9M [00:00<?, ?B/s]

--- Initializing Titans Neural Memory (dim=2880) ---
✅ Patched Neural Memory is online.


titan_reasoner_memory.pt:   0%|          | 0.00/70.8M [00:00<?, ?B/s]


✅ Titan-Reasoner is loaded and ready for evaluation.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
import os
import torch

def run_inference_on_dataframe(model, tokenizer, input_df, prompt_column, output_path):
    """Runs the Titan-Reasoner on each prompt in a DataFrame and saves the results."""
    results = []
    for index, row in tqdm(input_df.iterrows(), total=len(input_df), desc=f"Processing {prompt_column}"):
        prompt = row[prompt_column]
        
        messages = [{"role": "user", "content": prompt}]
        prompt_string = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = tokenizer(prompt_string, return_tensors="pt", max_length=2048, truncation=True).to("cuda")

        with torch.no_grad():
            # CORRECTED: Use do_sample=False for deterministic output
            outputs = model.base_model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response_text = tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        results.append(response_text)
        
    output_df = input_df.copy()
    output_df['titan_reasoner_output'] = results
    output_df.to_csv(output_path, index=False)
    print(f"✅ Inference complete. Results saved to {output_path}")
    return output_df

# --- Define the paths to your uploaded Kaggle dataset ---
dataset_path = "/kaggle/input/longmem/"
focused_csv_path = os.path.join(dataset_path, "cleaned_longmemeval_s_focused.csv")
full_csv_path = os.path.join(dataset_path, "cleaned_longmemeval_s_full.csv")

# --- Load the CSV files into pandas DataFrames ---
print("--- Loading dataframes for inference ---")
focused_df = pd.read_csv(focused_csv_path)
full_df = pd.read_csv(full_csv_path)
print("✅ Dataframes loaded successfully.")

# --- Create Results Directory ---
os.makedirs("results", exist_ok=True)

# --- Run on Focused Dataset ---
run_inference_on_dataframe(
    titan_reasoner_model, tokenizer, focused_df,
    'focused_prompt', 'results/titan_reasoner_focused_results.csv'
)

# --- Run on Full Dataset ---
run_inference_on_dataframe(
    titan_reasoner_model, tokenizer, full_df,
    'full_prompt', 'results/titan_reasoner_full_results.csv'
)

--- Loading dataframes for inference ---
✅ Dataframes loaded successfully.


Processing focused_prompt:   0%|          | 0/306 [00:00<?, ?it/s]

In [ ]:
# ==============================================================================
# Cell 5 (Final Version): Evaluate Results with Llama 3.1 as an Open-Source Judge
# ==============================================================================
import pandas as pd
import os
from unsloth import FastLanguageModel
import torch
from tqdm.notebook import tqdm

# --- Load the Open-Source Judge Model ---
# We use Llama 3.1 8B Instruct, a powerful and reliable open-source model.
# The 4-bit version is highly efficient for this task.
print("--- Loading the Open-Source Judge Model (Llama 3.1 8B Instruct) ---")
judge_model, judge_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3.1-8b-instruct-bnb-4bit",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(judge_model)
print("✅ Open-Source Judge Model is loaded and ready.")

def judge_results_with_open_source(input_path, output_path):
    """Uses Llama 3.1 8B to judge the correctness of the model's outputs."""
    input_df = pd.read_csv(input_path)
    judge_scores = []
    
    judge_prompt_template = """
    Given a question, the correct answer, and a model's response, determine if the response is correct. 
    A correct response factually aligns with the correct answer.
    Respond with only the word "true" if the response is correct, or "false" if it is not. Do not provide any explanation.

    Question: {question}
    CORRECT answer: {correct_answer}
    Response to judge: {output}
    """
    
    for index, row in tqdm(input_df.iterrows(), total=len(input_df), desc=f"Judging {input_path}"):
        prompt = judge_prompt_template.format(
            question=row['question'],
            correct_answer=row['answer'],
            output=row['titan_reasoner_output']
        )
        
        messages = [{"role": "user", "content": prompt}]
        # The tokenizer will correctly apply the Llama 3.1 chat template
        input_text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = judge_tokenizer(input_text, return_tensors="pt").to("cuda")

        try:
            with torch.no_grad():
                outputs = judge_model.generate(
                    **inputs,
                    max_new_tokens=5, # We only need one word
                    temperature=0.0,  # Deterministic
                    pad_token_id=judge_tokenizer.eos_token_id
                )
            
            result = judge_tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).lower().strip()
            judge_scores.append(1 if 'true' in result else 0)

        except Exception as e:
            print(f"Error judging row {index}: {e}")
            judge_scores.append(0) # Default to incorrect on error
            
    output_df = input_df.copy()
    output_df['llm_judge_score'] = judge_scores
    output_df.to_csv(output_path, index=False)
    print(f"✅ Judging complete. Evaluated results saved to {output_path}")

# --- Judge Both Result Files using Llama 3.1 ---
print("\\n--- Starting Evaluation with Llama 3.1 as Judge ---")
judge_results_with_open_source('results/titan_reasoner_focused_results.csv', 'results/titan_reasoner_focused_evaluated.csv')
judge_results_with_open_source('results/titan_reasoner_full_results.csv', 'results/titan_reasoner_full_evaluated.csv')

# --- Clean up the judge model from memory before visualizing ---
del judge_model, judge_tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()
print("\\n✅ Judge model unloaded from memory.")

In [ ]:
import matplotlib.pyplot as plt

def visualize_longmemeval_results(focused_filepath, full_filepath, model_name, output_path):
    """Creates and saves a bar chart comparing focused and full RAG performance."""
    focused_df = pd.read_csv(focused_filepath)
    full_df = pd.read_csv(full_filepath)
    
    focused_mean = focused_df['llm_judge_score'].mean()
    full_mean = full_df['llm_judge_score'].mean()
    
    print(f"--- {model_name} Performance ---")
    print(f"Focused Input (Reasoning Only): {focused_mean:.2%}")
    print(f"Full Input (Retrieval + Reasoning): {full_mean:.2%}")
    print(f"Performance Drop (Context Burden): {focused_mean - full_mean:.2%}")
    
    plt.figure(figsize=(8, 6))
    bars = plt.bar(['Focused', 'Full'], [focused_mean, full_mean], color=['#4CAF50', '#2196F3'])
    plt.ylim(0, 1)
    plt.ylabel('Accuracy (Judged by GPT-4o)')
    plt.title(f'LongMemEval Performance: {model_name}')
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2%}", ha='center', va='bottom', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    print(f"\\n✅ Visualization saved to: {output_path}")

# --- Generate the Final Plot ---
visualize_longmemeval_results(
    'results/titan_reasoner_focused_evaluated.csv',
    'results/titan_reasoner_full_evaluated.csv',
    "Titan-Reasoner (gpt-oss-20b)",
    'results/titan_reasoner_longmemeval_performance.png'
)